# 10 - Análise de Negócio

## Pergunta central

Em que medida os conteúdos que ganharam relevância digital no Brasil
em 2025 estão alinhados aos tipos de conteúdo consumidos por diferentes
faixas etárias?

## Perguntas analíticas

1. Quais categorias apresentam maior consumo em cada faixa etária?
2. Quais categorias apresentaram maior força digital em 2025?
3. O ranking de consumo de cada faixa etária está alinhado ao ranking
   de tendência digital?
4. Quais categorias apresentam maior alinhamento ou desalinhamento?
5. A conclusão se mantém quando ampliamos a comparação de cinco para
   oito categorias utilizando CETIC e YouTube?

## Estratégia

A análise principal utiliza cinco categorias presentes simultaneamente
nas três fontes:

- Notícias
- Esportes
- Música
- Humor
- Games

Como os indicadores das fontes possuem naturezas diferentes, a
comparação de alinhamento utiliza principalmente rankings relativos.

Também é apresentada uma análise ampliada com oito categorias,
utilizando CETIC e YouTube.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [0]:
df_perfil = spark.table(
    "workspace.mvp_gold.perfil_geracional_cetic_2025"
)

df_digital = spark.table(
    "workspace.mvp_gold.tendencias_digitais_nucleo_2025"
)

df_youtube = spark.table(
    "workspace.mvp_gold.tendencias_youtube_2025"
)

df_trends = spark.table(
    "workspace.mvp_gold.interesse_google_2025"
)

print("Perfil CETIC:", df_perfil.count())
print("Tendência digital núcleo:", df_digital.count())
print("Gold YouTube:", df_youtube.count())
print("Gold Google Trends:", df_trends.count())

In [0]:
display(
    df_perfil
    .filter(
        F.col("ranking_categoria_na_faixa") == 1
    )
    .select(
        "ordem_faixa_etaria",
        "faixa_etaria",
        "categoria",
        "percentual_consumo"
    )
    .orderBy(
        "ordem_faixa_etaria"
    )
)

In [0]:
display(
    df_perfil
    .select(
        "ordem_faixa_etaria",
        "faixa_etaria",
        "categoria",
        "percentual_consumo",
        "ranking_categoria_na_faixa"
    )
    .orderBy(
        "ordem_faixa_etaria",
        "ranking_categoria_na_faixa"
    )
)

In [0]:
display(
    df_perfil
    .select(
        "categoria",
        "ordem_faixa_etaria",
        "faixa_etaria",
        "percentual_consumo"
    )
    .orderBy(
        "categoria",
        "ordem_faixa_etaria"
    )
)

In [0]:
display(
    df_digital
    .select(
        "categoria",
        "participacao_videos_pct",
        "indice_medio",
        "score_youtube_0_100",
        "score_google_0_100",
        "score_tendencia_digital",
        "ranking_tendencia_digital"
    )
    .orderBy(
        "ranking_tendencia_digital"
    )
)

In [0]:
display(
    df_digital
    .select(
        "categoria",
        "score_tendencia_digital"
    )
    .orderBy(
        F.desc("score_tendencia_digital")
    )
)

In [0]:
categorias_nucleo = [
    "Notícias",
    "Esportes",
    "Música",
    "Humor",
    "Games"
]

df_perfil_nucleo = (
    df_perfil
    .filter(
        F.col("categoria").isin(
            categorias_nucleo
        )
    )
)

In [0]:
print(
    "Registros CETIC núcleo:",
    df_perfil_nucleo.count()
)

In [0]:
janela_cetic_nucleo = (
    Window
    .partitionBy(
        "faixa_etaria"
    )
    .orderBy(
        F.desc("percentual_consumo"),
        F.asc("categoria")
    )
)

df_perfil_nucleo = (
    df_perfil_nucleo
    .withColumn(
        "ranking_consumo_nucleo",
        F.row_number().over(
            janela_cetic_nucleo
        )
    )
)

In [0]:
display(
    df_perfil_nucleo
    .select(
        "ordem_faixa_etaria",
        "faixa_etaria",
        "categoria",
        "percentual_consumo",
        "ranking_consumo_nucleo"
    )
    .orderBy(
        "ordem_faixa_etaria",
        "ranking_consumo_nucleo"
    )
)

In [0]:
df_alinhamento = (
    df_perfil_nucleo
    .join(
        df_digital.select(
            "categoria",
            "participacao_videos_pct",
            "indice_medio",
            "score_youtube_0_100",
            "score_google_0_100",
            "score_tendencia_digital",
            "ranking_tendencia_digital"
        ),
        on="categoria",
        how="inner"
    )
)

In [0]:
print(
    "Combinações de alinhamento:",
    df_alinhamento.count()
)

In [0]:
df_alinhamento = (
    df_alinhamento
    .withColumn(
        "diferenca_ranking",
        F.abs(
            F.col("ranking_consumo_nucleo")
            -
            F.col("ranking_tendencia_digital")
        )
    )
)

In [0]:
df_alinhamento = (
    df_alinhamento
    .withColumn(
        "nivel_alinhamento",
        F.when(
            F.col("diferenca_ranking") == 0,
            "Alto"
        )
        .when(
            F.col("diferenca_ranking") == 1,
            "Próximo"
        )
        .otherwise(
            "Baixo"
        )
    )
)

In [0]:
display(
    df_alinhamento
    .select(
        "ordem_faixa_etaria",
        "faixa_etaria",
        "categoria",
        "percentual_consumo",
        "ranking_consumo_nucleo",
        "score_tendencia_digital",
        "ranking_tendencia_digital",
        "diferenca_ranking",
        "nivel_alinhamento"
    )
    .orderBy(
        "ordem_faixa_etaria",
        "ranking_consumo_nucleo"
    )
)

In [0]:
df_correlacao_faixa = (
    df_alinhamento
    .groupBy(
        "ordem_faixa_etaria",
        "faixa_etaria"
    )
    .agg(
        F.corr(
            "ranking_consumo_nucleo",
            "ranking_tendencia_digital"
        ).alias(
            "correlacao_ranking"
        ),

        F.round(
            F.avg(
                "diferenca_ranking"
            ),
            2
        ).alias(
            "diferenca_media_ranking"
        ),

        F.sum(
            F.when(
                F.col("diferenca_ranking") == 0,
                1
            ).otherwise(0)
        ).alias(
            "categorias_mesma_posicao"
        )
    )
)

In [0]:
df_correlacao_faixa = (
    df_correlacao_faixa
    .withColumn(
        "correlacao_ranking",
        F.round(
            "correlacao_ranking",
            3
        )
    )
    .withColumn(
        "interpretacao_alinhamento",
        F.when(
            F.col("correlacao_ranking") >= 0.70,
            "Alinhamento forte"
        )
        .when(
            F.col("correlacao_ranking") >= 0.40,
            "Alinhamento moderado"
        )
        .when(
            F.col("correlacao_ranking") >= 0.10,
            "Alinhamento fraco"
        )
        .when(
            F.col("correlacao_ranking") > -0.10,
            "Sem relação clara"
        )
        .otherwise(
            "Relação inversa"
        )
    )
)

In [0]:
display(
    df_correlacao_faixa
    .orderBy(
        "ordem_faixa_etaria"
    )
)

In [0]:
display(
    df_correlacao_faixa
    .orderBy(
        F.desc("correlacao_ranking")
    )
)

In [0]:
display(
    df_alinhamento
    .select(
        "faixa_etaria",
        "categoria",
        "ranking_consumo_nucleo",
        "ranking_tendencia_digital",
        "diferenca_ranking"
    )
    .orderBy(
        F.desc("diferenca_ranking"),
        "faixa_etaria"
    )
)

In [0]:
df_alinhamento_categoria = (
    df_alinhamento
    .groupBy(
        "categoria"
    )
    .agg(
        F.round(
            F.avg(
                "diferenca_ranking"
            ),
            2
        ).alias(
            "diferenca_media_ranking"
        ),

        F.sum(
            F.when(
                F.col("diferenca_ranking") == 0,
                1
            ).otherwise(0)
        ).alias(
            "faixas_mesma_posicao"
        )
    )
)

In [0]:
display(
    df_alinhamento_categoria
    .orderBy(
        "diferenca_media_ranking"
    )
)

In [0]:
categorias_ampliadas = [
    "Notícias",
    "Esportes",
    "Música",
    "Humor",
    "Games",
    "Animações",
    "Tutoriais / Educação",
    "Influenciadores"
]

df_youtube_ampliado = (
    df_youtube
    .filter(
        F.col("categoria").isin(
            categorias_ampliadas
        )
    )
)

In [0]:
print(
    "Categorias YouTube ampliadas:",
    df_youtube_ampliado
    .select("categoria")
    .distinct()
    .count()
)

In [0]:
janela_youtube_ampliado = (
    Window
    .orderBy(
        F.desc("videos_unicos"),
        F.asc("categoria")
    )
)

df_youtube_ampliado = (
    df_youtube_ampliado
    .withColumn(
        "ranking_youtube_ampliado",
        F.row_number().over(
            janela_youtube_ampliado
        )
    )
)

In [0]:
janela_cetic_ampliado = (
    Window
    .partitionBy(
        "faixa_etaria"
    )
    .orderBy(
        F.desc("percentual_consumo"),
        F.asc("categoria")
    )
)

df_cetic_ampliado = (
    df_perfil
    .filter(
        F.col("categoria").isin(
            categorias_ampliadas
        )
    )
    .withColumn(
        "ranking_consumo_ampliado",
        F.row_number().over(
            janela_cetic_ampliado
        )
    )
)

In [0]:
df_alinhamento_ampliado = (
    df_cetic_ampliado
    .join(
        df_youtube_ampliado.select(
            "categoria",
            "videos_unicos",
            "participacao_videos_pct",
            "ranking_youtube_ampliado"
        ),
        on="categoria",
        how="inner"
    )
    .withColumn(
        "diferenca_ranking",
        F.abs(
            F.col("ranking_consumo_ampliado")
            -
            F.col("ranking_youtube_ampliado")
        )
    )
)

In [0]:
print(
    "Combinações ampliadas:",
    df_alinhamento_ampliado.count()
)

In [0]:
df_correlacao_ampliada = (
    df_alinhamento_ampliado
    .groupBy(
        "ordem_faixa_etaria",
        "faixa_etaria"
    )
    .agg(
        F.round(
            F.corr(
                "ranking_consumo_ampliado",
                "ranking_youtube_ampliado"
            ),
            3
        ).alias(
            "correlacao_ranking_cetic_youtube"
        ),

        F.round(
            F.avg(
                "diferenca_ranking"
            ),
            2
        ).alias(
            "diferenca_media_ranking"
        )
    )
)

In [0]:
display(
    df_correlacao_ampliada
    .orderBy(
        "ordem_faixa_etaria"
    )
)

In [0]:
df_trends_semanal = spark.table(
    "workspace.mvp_silver.google_trends_2025"
)

In [0]:
janela_pico = (
    Window
    .partitionBy("categoria")
    .orderBy(
        F.desc("indice_trends"),
        F.asc("data_semana")
    )
)

df_picos_trends = (
    df_trends_semanal
    .withColumn(
        "_ordem",
        F.row_number().over(
            janela_pico
        )
    )
    .filter(
        F.col("_ordem") == 1
    )
    .select(
        "categoria",
        "data_semana",
        "indice_trends"
    )
)

In [0]:
display(
    df_picos_trends
    .orderBy(
        F.desc("indice_trends")
    )
)

In [0]:
display(
    df_trends_semanal
    .select(
        "data_semana",
        "categoria",
        "indice_trends"
    )
    .orderBy(
        "data_semana"
    )
)

In [0]:
(
    df_alinhamento.write
    .format("delta")
    .mode("overwrite")
    .option(
        "overwriteSchema",
        "true"
    )
    .saveAsTable(
        "workspace.mvp_gold.alinhamento_geracional_2025"
    )
)

In [0]:
(
    df_correlacao_faixa.write
    .format("delta")
    .mode("overwrite")
    .option(
        "overwriteSchema",
        "true"
    )
    .saveAsTable(
        "workspace.mvp_gold.resumo_alinhamento_geracional_2025"
    )
)

In [0]:
(
    df_alinhamento_ampliado.write
    .format("delta")
    .mode("overwrite")
    .option(
        "overwriteSchema",
        "true"
    )
    .saveAsTable(
        "workspace.mvp_gold.alinhamento_cetic_youtube_ampliado_2025"
    )
)

In [0]:
%sql

SHOW TABLES IN workspace.mvp_gold;

In [0]:
print(
    "Perfil CETIC:",
    df_perfil.count()
)

print(
    "Núcleo digital:",
    df_digital.count()
)

print(
    "Cruzamento principal:",
    df_alinhamento.count()
)

print(
    "Resumo por faixa:",
    df_correlacao_faixa.count()
)

print(
    "Cruzamento ampliado:",
    df_alinhamento_ampliado.count()
)

## Critério de alinhamento

Os percentuais de consumo da TIC Domicílios, a participação de vídeos
do YouTube e os índices do Google Trends possuem unidades e processos
de geração distintos.

Por esse motivo, os valores absolutos dessas fontes não foram
interpretados como diretamente equivalentes.

A análise principal de alinhamento considera a posição relativa das
cinco categorias comuns em cada fonte.

Para cada faixa etária, o ranking de consumo observado no CETIC foi
comparado ao ranking de força digital construído a partir de YouTube
Trending e Google Trends.

Foram utilizados:

- diferença absoluta entre posições no ranking;
- correlação entre rankings por faixa etária;
- quantidade de categorias que ocupam a mesma posição.

A análise ampliada repete a lógica para oito categorias utilizando
CETIC e YouTube.

Os resultados representam associação entre padrões observados nas
fontes e não demonstram relação causal entre idade e viralização.

# Principais conclusões

## 1. Perfil geracional de consumo

A preencher a partir dos resultados observados nas tabelas e gráficos.

## 2. Tendências digitais de 2025

A preencher a partir do ranking digital consolidado.

## 3. Alinhamento por faixa etária

A preencher a partir da correlação e diferença entre rankings.

## 4. Categorias com maior alinhamento e desalinhamento

A preencher a partir do cruzamento por categoria.

## 5. Análise ampliada CETIC × YouTube

A preencher a partir da comparação das oito categorias.

## Limitações

- As fontes possuem metodologias e unidades distintas.
- Google Trends mede interesse relativo de busca, e não volume absoluto.
- YouTube Trending representa relevância na plataforma e não o consumo
  digital total da população brasileira.
- A TIC Domicílios representa comportamento declarado dos entrevistados.
- O score digital é um indicador analítico construído especificamente
  para este MVP.
- Os resultados indicam associação e não causalidade.